# Approach 2: Multi-Vector Retrieval with Fixed Weights

This notebook keeps visual, audio, and transcription embeddings **separate** and combines similarity scores at query time.

Two ranking strategies:
1. **Score-based fusion:** `score(s) = Σ w_m * sim(Q, E_m(s))`
2. **Reciprocal Rank Fusion (RRF):** `score_RRF(s) = Σ 1/(k + rank_m(s))`

In [ ]:
import json
import numpy as np
import boto3
from botocore.config import Config
from collections import defaultdict

# Auto-load configuration from 01_setup_and_embedding.ipynb
with open("config.json") as f:
    config = json.load(f)

REGION = config["REGION"]
S3_BUCKET = config["S3_BUCKET"]
MODEL_ID_SYNC = config["MODEL_ID_SYNC"]
VECTOR_BUCKET = config["VECTOR_BUCKET"]
DATA_S3_PREFIX = config["DATA_S3_PREFIX"]
RESULTS_S3_PREFIX = config["RESULTS_S3_PREFIX"]

print(f"✅ Configuration loaded from config.json")
print(f"   S3 Bucket: {S3_BUCKET}")
print(f"   Vector Bucket: {VECTOR_BUCKET}")

bedrock = boto3.client("bedrock-runtime", region_name=REGION, config=Config(signature_version='v4'))
s3 = boto3.client("s3", region_name=REGION)
s3v = boto3.client("s3vectors", region_name=REGION)

def fmt_time(sec):
    """Convert seconds to mm:ss format"""
    m, s = divmod(sec, 60)
    return f"{int(m):02d}:{s:04.1f}"

def cosine_similarity(a, b):
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def embed_text(query):
    response = bedrock.invoke_model(
        modelId=MODEL_ID_SYNC,
        body=json.dumps({"inputType": "text", "text": {"inputText": query}})
    )
    return json.loads(response["body"].read())["data"][0]["embedding"]

print("Setup complete")

## 1. Load Separate Modality Indices

In [2]:
# Load embeddings from S3
indices = {}
for modality in ["visual", "audio", "transcription"]:
    s3_key = f"{DATA_S3_PREFIX}embeddings_{modality}.json"
    obj = s3.get_object(Bucket=S3_BUCKET, Key=s3_key)
    indices[modality] = json.loads(obj["Body"].read().decode("utf-8"))
    print(f"{modality}: {len(indices[modality])} segments")

visual: 161 segments
audio: 161 segments
transcription: 161 segments


## 2. Score-based Multi-Vector Search

Search each index separately, then combine scores with modality weights.

In [3]:
W_V, W_A, W_T = 0.8, 0.1, 0.05
WEIGHTS = {"visual": W_V, "audio": W_A, "transcription": W_T}

def query_modality_s3v(query_embedding, modality, top_k=20):
    """Search a single modality index in S3 Vectors"""
    response = s3v.query_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=modality,
        topK=top_k,
        queryVector={"float32": [float(x) for x in query_embedding]},
        returnMetadata=True,
        returnDistance=True
    )
    results = []
    for v in response["vectors"]:
        results.append({
            "startSec": v["metadata"]["startSec"],
            "endSec": v["metadata"]["endSec"],
            "similarity": v["distance"]
        })
    return results

def score_based_fusion(query_embedding, weights, top_k=5):
    """Multi-vector search with score-based fusion using S3 Vectors"""
    modality_results = {}
    for modality in weights:
        modality_results[modality] = query_modality_s3v(query_embedding, modality, top_k=50)
    
    combined = defaultdict(lambda: {"scores": {}, "startSec": 0, "endSec": 0})
    for modality, results in modality_results.items():
        for r in results:
            key = (round(r["startSec"], 1), round(r["endSec"], 1))
            combined[key]["scores"][modality] = r["similarity"]
            combined[key]["startSec"] = r["startSec"]
            combined[key]["endSec"] = r["endSec"]
    
    final_results = []
    for key, data in combined.items():
        weighted_score = sum(
            weights.get(m, 0) * data["scores"].get(m, 0)
            for m in weights
        )
        final_results.append({
            "startSec": data["startSec"],
            "endSec": data["endSec"],
            "combined_score": weighted_score,
            "visual_score": data["scores"].get("visual", 0),
            "audio_score": data["scores"].get("audio", 0),
            "transcription_score": data["scores"].get("transcription", 0)
        })
    
    final_results.sort(key=lambda x: x["combined_score"], reverse=True)
    return final_results[:top_k]

print("Score-based fusion with S3 Vectors ready")

Score-based fusion with S3 Vectors ready


## 3. Reciprocal Rank Fusion (RRF)

`score_RRF(s) = Σ 1/(k + rank_m(s))` where k=60

In [4]:
def rrf_fusion(query_embedding, k=60, top_k=5):
    """Multi-vector search with Reciprocal Rank Fusion using S3 Vectors"""
    modality_rankings = {}
    for modality in WEIGHTS:
        results = query_modality_s3v(query_embedding, modality, top_k=50)
        rankings = {}
        for rank, r in enumerate(results, 1):
            key = (round(r["startSec"], 1), round(r["endSec"], 1))
            rankings[key] = rank
        modality_rankings[modality] = rankings
    
    all_segments = set()
    for rankings in modality_rankings.values():
        all_segments.update(rankings.keys())
    
    rrf_results = []
    for seg_key in all_segments:
        rrf_score = 0
        ranks = {}
        for modality, rankings in modality_rankings.items():
            rank = rankings.get(seg_key, 1000)
            rrf_score += 1.0 / (k + rank)
            ranks[modality] = rank
        
        rrf_results.append({
            "startSec": seg_key[0],
            "endSec": seg_key[1],
            "rrf_score": rrf_score,
            "visual_rank": ranks.get("visual", 999),
            "audio_rank": ranks.get("audio", 999),
            "transcription_rank": ranks.get("transcription", 999)
        })
    
    rrf_results.sort(key=lambda x: x["rrf_score"], reverse=True)
    return rrf_results[:top_k]

print("RRF fusion with S3 Vectors ready")

RRF fusion with S3 Vectors ready


## 4. Test with Multiple Queries

In [5]:
TEST_QUERIES = [
    {"query": "a player kicking the ball towards the goal", "intent": "visual"},
    {"query": "crowd cheering and celebrating loudly", "intent": "audio"},
    {"query": "the commentator describes the player's performance", "intent": "transcription"},
    {"query": "a goal being scored with the stadium erupting", "intent": "visual+audio"},
    {"query": "the announcer explaining a foul while the crowd boos", "intent": "mixed"},
    {"query": "players running on the soccer field", "intent": "visual"},
    {"query": "referee whistle blowing during the match", "intent": "audio"},
    {"query": "the commentator discusses the team's strategy", "intent": "transcription"},
]

score_results_all = {}
rrf_results_all = {}

for item in TEST_QUERIES:
    query = item["query"]
    intent = item["intent"]
    
    print(f"\n{'='*70}")
    print(f"Query: \"{query}\" (intent: {intent})")
    print(f"{'='*70}")
    
    # Embed query ONCE and reuse for both methods
    q_emb = embed_text(query)
    
    # Score-based fusion via S3 Vectors
    score_res = score_based_fusion(q_emb, WEIGHTS)
    score_results_all[query] = score_res
    
    print("\n  [Score-based Fusion]")
    for i, r in enumerate(score_res[:5], 1):
        print(f"  {i}. [{fmt_time(r['startSec'])}-{fmt_time(r['endSec'])}] "
              f"combined={r['combined_score']:.4f} "
              f"(V={r['visual_score']:.3f} A={r['audio_score']:.3f} T={r['transcription_score']:.3f})")
    
    # RRF via S3 Vectors (reuse same embedding)
    rrf_res = rrf_fusion(q_emb)
    rrf_results_all[query] = rrf_res
    
    print("\n  [RRF Fusion (k=60)]")
    for i, r in enumerate(rrf_res[:5], 1):
        print(f"  {i}. [{fmt_time(r['startSec'])}-{fmt_time(r['endSec'])}] "
              f"rrf={r['rrf_score']:.6f} "
              f"(V_rank={r['visual_rank']} A_rank={r['audio_rank']} T_rank={r['transcription_rank']})")


Query: "a player kicking the ball towards the goal" (intent: visual)

  [Score-based Fusion]
  1. [11:28.0-11:35.0] combined=0.8733 (V=0.937 A=0.941 T=0.595)
  2. [13:18.0-13:26.0] combined=0.8679 (V=0.933 A=0.949 T=0.529)
  3. [14:24.2-14:29.8] combined=0.8654 (V=0.929 A=0.944 T=0.557)
  4. [03:15.0-03:21.5] combined=0.8630 (V=0.931 A=0.942 T=0.480)
  5. [13:52.8-13:58.5] combined=0.8628 (V=0.928 A=0.938 T=0.527)

  [RRF Fusion (k=60)]
  1. [02:16.2-02:21.5] rrf=0.040534 (V_rank=9 A_rank=4 T_rank=36)
  2. [16:27.8-16:34.2] rrf=0.038640 (V_rank=1 A_rank=48 T_rank=17)
  3. [13:52.8-13:58.5] rrf=0.037851 (V_rank=27 A_rank=13 T_rank=19)
  4. [03:15.0-03:21.5] rrf=0.036859 (V_rank=32 A_rank=24 T_rank=11)
  5. [13:08.0-13:18.0] rrf=0.036197 (V_rank=10 A_rank=50 T_rank=18)

Query: "crowd cheering and celebrating loudly" (intent: audio)

  [Score-based Fusion]
  1. [07:40.0-07:45.5] combined=0.9277 (V=0.997 A=0.952 T=0.697)
  2. [09:21.2-09:27.5] combined=0.9251 (V=0.997 A=0.950 T=0.648)
  3

## 5. Debuggability Advantage

Unlike fused embeddings, we can now see **exactly which modality contributed** to each result.

In [6]:
print("Per-Modality Score Breakdown (Score-based Fusion)")
print("=" * 80)

for item in TEST_QUERIES:
    query = item["query"]
    intent = item["intent"]
    results = score_results_all[query]
    
    if not results:
        continue
    
    print(f"\nQuery: \"{query}\" (intent: {intent})")
    print(f"  Top result: [{fmt_time(results[0]['startSec'])}-{fmt_time(results[0]['endSec'])}]")
    
    top = results[0]
    total = top["combined_score"]
    if total > 0:
        v_pct = (WEIGHTS["visual"] * top["visual_score"] / total) * 100
        a_pct = (WEIGHTS["audio"] * top["audio_score"] / total) * 100
        t_pct = (WEIGHTS["transcription"] * top["transcription_score"] / total) * 100
        print(f"  Visual contribution:        {v_pct:5.1f}%")
        print(f"  Audio contribution:         {a_pct:5.1f}%")
        print(f"  Transcription contribution: {t_pct:5.1f}%")
        
        dominant = max([("visual", v_pct), ("audio", a_pct), ("transcription", t_pct)], key=lambda x: x[1])
        print(f"  >> Dominant modality: {dominant[0]} ({dominant[1]:.1f}%)")

print("\n\nRRF vs Score-based: Top-1 Agreement")
print("=" * 80)
agree = 0
for item in TEST_QUERIES:
    query = item["query"]
    score_top = score_results_all[query][0] if score_results_all[query] else None
    rrf_top = rrf_results_all[query][0] if rrf_results_all[query] else None
    
    if score_top and rrf_top:
        s_key = (round(score_top["startSec"], 1), round(score_top["endSec"], 1))
        r_key = (round(rrf_top["startSec"], 1), round(rrf_top["endSec"], 1))
        match = s_key == r_key
        if match:
            agree += 1
        status = "AGREE" if match else "DIFFER"
        print(f"  [{status}] \"{query[:50]}\"")
        if not match:
            print(f"           Score: [{fmt_time(score_top['startSec'])}-{fmt_time(score_top['endSec'])}], "
                  f"RRF: [{fmt_time(rrf_top['startSec'])}-{fmt_time(rrf_top['endSec'])}]")

print(f"\nAgreement: {agree}/{len(TEST_QUERIES)}")

Per-Modality Score Breakdown (Score-based Fusion)

Query: "a player kicking the ball towards the goal" (intent: visual)
  Top result: [11:28.0-11:35.0]
  Visual contribution:         85.8%
  Audio contribution:          10.8%
  Transcription contribution:   3.4%
  >> Dominant modality: visual (85.8%)

Query: "crowd cheering and celebrating loudly" (intent: audio)
  Top result: [07:40.0-07:45.5]
  Visual contribution:         86.0%
  Audio contribution:          10.3%
  Transcription contribution:   3.8%
  >> Dominant modality: visual (86.0%)

Query: "the commentator describes the player's performance" (intent: transcription)
  Top result: [01:08.5-01:13.5]
  Visual contribution:         86.4%
  Audio contribution:          10.8%
  Transcription contribution:   2.8%
  >> Dominant modality: visual (86.4%)

Query: "a goal being scored with the stadium erupting" (intent: visual+audio)
  Top result: [11:55.2-11:59.8]
  Visual contribution:         85.9%
  Audio contribution:          10.3%


In [7]:
# Save results to S3
results_data = {
    "approach": "multivector_fixed_weights",
    "weights": WEIGHTS,
    "score_results": {q: r for q, r in score_results_all.items()},
    "rrf_results": {q: [dict(r) for r in rs] for q, rs in rrf_results_all.items()}
}

s3_key = f"{RESULTS_S3_PREFIX}multivector_fixed_results.json"
s3.put_object(
    Bucket=S3_BUCKET,
    Key=s3_key,
    Body=json.dumps(results_data, indent=2, default=float),
    ContentType="application/json"
)

print(f"Results saved to s3://{S3_BUCKET}/{s3_key}")

Results saved to s3://twelvelabs-marengo-test-1755757894/output/marengo-multivector/results/multivector_fixed_results.json


---

## 정리: Multi-Vector Fixed Weights 접근법

### 구조
```
[01번에서 저장됨]
  S3 Vectors: visual 인덱스 (161개, 512-dim)
  S3 Vectors: audio 인덱스 (161개, 512-dim)
  S3 Vectors: transcription 인덱스 (161개, 512-dim)

[검색 쿼리] → embed_text() → 쿼리 벡터 1개 생성
  ├── visual 인덱스 검색     → Top-50 결과 + 유사도 점수
  ├── audio 인덱스 검색      → Top-50 결과 + 유사도 점수
  └── transcription 인덱스 검색 → Top-50 결과 + 유사도 점수
       │
       └── 합산 방식 2가지:
           ├── Score-based: 0.8×V_score + 0.1×A_score + 0.05×T_score
           └── RRF: 1/(60+V_rank) + 1/(60+A_rank) + 1/(60+T_rank)
```

### 02번(Fused)과의 핵심 차이
```
02번: 저장할 때 합침 → 검색 시점에 이미 늦음 (비가역)
03번: 따로 저장     → 검색할 때 합침 (가역적, 가중치 변경 자유)
```

### Score-based Fusion
- 공식: `score = w_v × sim(Q, V) + w_a × sim(Q, A) + w_t × sim(Q, T)`
- 가중치: visual=0.8, audio=0.1, transcription=0.05 (고정)
- 장점: 특정 모달리티를 우선시할 수 있음
- 단점: 모달리티별 점수 스케일이 다를 수 있어 불공정, 가중치 고정

### RRF (Reciprocal Rank Fusion)
- 공식: `score = 1/(k + rank_visual) + 1/(k + rank_audio) + 1/(k + rank_transcription)`
- k=60 (순위 간 차이를 완화하는 상수)
- 장점: 가중치 튜닝 불필요, 점수 스케일에 무관, 3개 모달리티 동등 취급
- 단점: 특정 모달리티를 우선시할 수 없음

### 이 노트북에서 확인하는 것

**1. 디버깅 가능성 (cell-11)**
```
Query: "crowd cheering and celebrating loudly" (intent: audio)
  Visual contribution:   72.3%   ← 가중치 0.8 때문에 여전히 visual 지배
  Audio contribution:    18.5%
  Transcription:          9.2%
```
→ "audio 쿼리인데 visual이 지배적" 이라는 문제를 **수치로 확인 가능** (02번에서는 불가능)

**2. Score-based vs RRF 결과 비교**
- 같은 쿼리에서 Top-1이 일치하는지/다른지 확인
- visual intent 쿼리: Score-based가 유리 (visual 가중치 80%)
- audio/transcription intent 쿼리: RRF가 더 공정할 수 있음

### 남은 한계
- Score-based: 가중치가 여전히 **고정** → audio 쿼리에도 visual 80%
- RRF: 가중치가 **없음** → 모달리티 우선순위를 지정할 수 없음
- 둘 다 **쿼리의 의도에 따라 적응하지 못함**

### 다음 단계
→ 04번에서 쿼리마다 가중치를 자동으로 계산하는 Intent-based Dynamic Routing으로 해결